# Lesson 12 — Two probabilities do not determine the probability of both

A parlay pays only when every leg lands, and the exchange states its legs in the market metadata. What the legs pin down is not a price but a band, and the width of that band is how far the parlay can move with no leg moving at all.

**The rule.** `max(0, Σpᵢ − (n−1)) ≤ P(all) ≤ min pᵢ`

**When it holds.** For any conjunction whose legs the venue lists, whatever the dependence between them turns out to be.

**When it fails.** Treating the independence product as a fair value. Legs are routinely dependent, and a price above Πpᵢ is not evidence of anything on its own; parlays are quoted one-sided, so the reading is taken from the offer and carries the maker's margin with it.

| | |
|---|---|
| Lesson id | `frechet` |
| Pane it appears on | `combos` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/frechet.py`, `modules/coherence/drivers/kalshi_combos.py` |
| Tests that go red if it stops being true | `tests/test_coherence_frechet.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The venue states the conjunction

In [ ]:
from modules.coherence.drivers.kalshi_combos import parse_combo
from modules.coherence.drivers.kalshi_parse import ParseError
from modules.coherence.kernel.book import Book, Level
from modules.coherence.kernel.frechet import assess, rows_for_combo, side_label

# The venue states the conjunction. Unlike every other relation in this engine,
# nothing here is inferred from titles or strikes.
PAYLOAD = {
    "ticker": "KXPARLAY-26AUG23-ABC",
    "mve_collection_ticker": "KXPARLAY",
    "exchange_index": 1,
    "yes_sub_title": "City drop points, Liverpool win, and BTC stays under 110k",
    "mve_selected_legs": [
        {"market_ticker": "A", "event_ticker": "KXEPLGAME-CITY", "side": "yes"},
        {"market_ticker": "B", "event_ticker": "KXEPLGAME-LIV", "side": "yes"},
        {"market_ticker": "C", "event_ticker": "KXBTCD-110K", "side": "no"},
    ],
}

unknown_shards = parse_combo(PAYLOAD)
known_shards = parse_combo(PAYLOAD, shards={"A": 1, "B": 1, "C": 1})
print(f"  {unknown_shards.label}")
print(f"  {len(unknown_shards.legs)} legs, collection {unknown_shards.collection_ticker}")
for leg in unknown_shards.legs:
    print(f"    {leg.ticker} on {leg.event_ticker}, settling {leg.side} (opposite is {leg.opposite})")
print()
print(f"  shards unknown -> scope {unknown_shards.scope}")
print(f"  shards known   -> scope {known_shards.scope}")
print("  Unknown is treated as cross-shard, because assuming a leg is co-located when it")
print("  is not understates the legging risk, and understating it is the expensive error.")
print()
print(f"  a plain market with no mve_selected_legs parses to {parse_combo({'ticker': 'X'})!r}")
try:
    parse_combo({"ticker": "X", "mve_selected_legs": [{"market_ticker": "A", "side": ""}]})
except ParseError as exc:
    print(f"  a leg with no side is refused, not defaulted: {exc}")

## 2. The band the legs leave

In [ ]:
combo = known_shards


def quote(ticker, yes_bid, no_bid, size=20_000):
    return Book(
        ticker=ticker,
        yes_bids=(Level(Decimal(yes_bid), size),),
        no_bids=(Level(Decimal(no_bid), size),),
    )


# Legs A and B are YES legs quoted around 0.90 and 0.85. Leg C is a NO leg: its
# market's YES mid is 0.20, so the leg the parlay needs is worth 0.80.
BOOKS = {
    "A": quote("A", "0.8900", "0.0900"),
    "B": quote("B", "0.8400", "0.1400"),
    "C": quote("C", "0.1900", "0.7900"),
    combo.ticker: quote(combo.ticker, "0.3800", "0.6000"),
}
reading = assess(combo, BOOKS)

print("  leg                       settles   p       buy p    buy the opposite")
for row in reading.legs:
    print(f"  {row.label:<24}  {row.side:<7}   {row.probability}  {row.buy_cost}   {row.opposite_cost}")
print()
count = len(reading.legs)
total = sum((row.probability for row in reading.legs), Decimal(0))
print(f"  sum of leg probabilities  {total}   over n = {count} legs")
print(f"  lower bound  max(0, {total} - {count - 1})  =  {reading.lower_bound}")
print(f"  upper bound  min of the legs             =  {reading.upper_bound}")
print(f"  band width                               =  {reading.band_width}")
print(f"  independence would say                   =  {reading.independence.quantize(Decimal('0.0001'))}")
print()
print("  The upper bound is the conjunction being a subset of each leg. The lower is the")
print("  union bound rearranged: the legs can fail on disjoint futures only until the")
print("  failure probabilities exhaust the space.")

## 3. Quoted outside the band, which is the only mispricing

In [ ]:
print(f"  parlay bid {reading.combo_bid}, ask {reading.combo_ask}, mid {reading.combo_mid}")
print(f"  band       [{reading.lower_bound}, {reading.upper_bound}]")
print(f"  inside the band : {reading.inside_band}")
print(f"  dependence read : {reading.dependence}")
print()
print("  It is quoted BELOW the lower bound, which no dependence structure can produce.")
print("  That is the mispricing. Sitting below the independence product is not: legs are")
print("  routinely dependent, and where a price sits INSIDE the band is a statement about")
print("  dependence, which nothing on this exchange quotes.")

## 4. The cover, and why it cannot cost under a dollar

In [ ]:
rows = rows_for_combo(combo, BOOKS)
print(f"  {len(rows)} rows, all in the shape the solver already takes:")
for row in rows:
    label = "cover (lower bound)" if row.bound == Decimal(1) else "upper bound"
    print(f"    {label:<20} bound {row.bound}  cost {row.cost}  slack {row.slack}  violated {row.violated}")
print()
cover = next(row for row in rows if row.bound == Decimal(1))
print("  the cover portfolio, leg by leg:")
for leg in cover.legs:
    print(f"    {leg.direction} {leg.label:<34} side {leg.side}  @ {leg.price}")
print(f"    total cost {cover.cost}, against the dollar it is guaranteed to pay")
print()
print("  Buy the parlay and the opposite of every leg. If all three legs land the parlay")
print("  pays a dollar and the opposites pay nothing. If k >= 1 legs miss, the parlay pays")
print("  nothing and exactly k opposites pay a dollar each. So the set pays at least a")
print("  dollar in every future, and any total cost below one is a Dutch book.")

## 5. The opposite of a NO leg is a YES purchase

In [ ]:
no_leg = combo.legs[2]
no_reading = reading.legs[2]
print(f"  leg {no_leg.ticker} settles {no_leg.side}, so its opposite is {no_leg.opposite}")
print(f"    the leg itself : {side_label(no_reading.label, no_leg.side)}  costs {no_reading.buy_cost}")
print(f"    its opposite   : {side_label(no_reading.label, no_leg.opposite)}  costs {no_reading.opposite_cost}")
print()
cover_leg = next(leg for leg in cover.legs if leg.ticker == no_leg.ticker)
print(f"  and in the cover portfolio it appears as: {cover_leg.direction} {cover_leg.label} (side {cover_leg.side})")
print()
print("  The negation of a NO leg is a YES PURCHASE. Roughly half of Kalshi's parlay legs")
print("  are NO legs, so a label composed as 'not <market>' gets this backwards on exactly")
print("  the legs where it matters, and the order plan buys the opposite contract. Every")
print("  label the certificate prints therefore carries the side it settles on.")
print()
print(f"  scope with the shards supplied : {combo.scope}")
print(f"  scope with them unknown        : {unknown_shards.scope}")
print()
print("  The combo above is same-shard only because this notebook told parse_combo where")
print("  its legs live. Live, a parlay on one shard references markets on others, so the")
print("  real listing is cross-shard and carries the most expensive legging tier in")
print("  costs.py: order groups do not work across exchange instances, so nothing cancels")
print("  the rest of the group when one leg over-fills.")

## 6. The certificate

In [ ]:
from modules.coherence.kernel import closedform
from modules.coherence.kernel.costs import FeeSchedule
from modules.coherence.kernel.lattice import Component

shell = Component(
    component_id=combo.ticker, event_ticker=combo.ticker, series_ticker=combo.collection_ticker,
    exchange_index=combo.exchange_index, mutually_exclusive=False, nodes=[],
)
print(closedform.solve(shell, rows, FeeSchedule()).render_text())